# Compare predictions

Use pointwise log likelihoods to evaluate how a fitted model predicts held-out
observations with Pareto-smoothed importance sampling leave-one-out
cross-validation (PSIS-LOO). Start with {doc}`checked chains <goose-diagnostics>`;
LOO does not diagnose MCMC convergence.

In [1]:
import jax.numpy as jnp
import numpy as np
import tensorflow_probability.substrates.jax.bijectors as tfb
import tensorflow_probability.substrates.jax.distributions as tfd

import liesel.goose as gs
import liesel.model as lsl

## Prepare the example

These examples require the regression `model` and sampling `results` from
{doc}`tutorials/md/01c-transform`.

In [2]:
# Python cell fragment: imports are visible in each guide.
# Simulate regression observations.
rng = np.random.default_rng(42)
n = 500
true_beta = np.array([1.0, 2.0])
true_sigma = 1.0
x = rng.uniform(size=n)
X_mat = np.column_stack([np.ones(n), x])
y_vec = X_mat @ true_beta + rng.normal(scale=true_sigma, size=n)

# Define the priors.
beta = lsl.Var.new_param(
    jnp.zeros(2),
    dist=lsl.Dist(tfd.Normal, 0.0, 5.0),
    name="beta",
)
sigma_sq = lsl.Var.new_param(
    1.0,
    dist=lsl.Dist(tfd.InverseGamma, concentration=3.0, scale=2.0),
    name="sigma_sq",
)

# Connect the parameters to the observations.
sigma = lsl.Var.new_calc(jnp.sqrt, sigma_sq, name="sigma")
X = lsl.Var.new_obs(X_mat, name="X")
mu = lsl.Var.new_calc(jnp.dot, X, beta, name="mu")
y = lsl.Var.new_obs(
    y_vec,
    dist=lsl.Dist(tfd.Normal, mu, sigma),
    name="y",
)

sigma_sq.biject(tfb.Exp(), name="log_sigma_sq")
log_sigma_sq = sigma_sq.bijected_var

joint = gs.MCMCSpec(
    gs.NUTSKernel,
    kernel_group="regression",
    jitter_dist=tfd.Normal(0.0, 0.2),
)
beta.inference = joint
log_sigma_sq.inference = joint
model = lsl.Model(y)


In [3]:
results = gs.LieselMCMC(model).run_for_epochs(
    seed=1,
    num_chains=4,
    adaptation=1000,
    posterior=1000,
    positions_included=["sigma_sq"],
    show_progress=False,
)

liesel.goose.engine - INFO - Initializing kernels...


liesel.goose.engine - INFO - Done


liesel.goose.engine - INFO - Finished warmup


## Compute log likelihoods

For `model` and `results` from {doc}`tutorials/md/01c-transform`:

In [4]:
samples = results.get_posterior_samples()
log_lik = lsl.log_prob_pointwise({"y": model.vars["y"]}, samples)
log_lik_y = log_lik["y_log_prob"]
loo_result = gs.loo(log_lik_y)

In [5]:
log_lik_y.shape

(4, 1000, 500)

In [6]:
loo_result

Computed from 4000 posterior samples and 500 observations log-likelihood matrix.

         Estimate       SE
elpd_loo  -721.24    16.56
p_loo        3.09        -
------

Pareto k diagnostic values:
                         Count   Pct.
(-Inf, 0.70]   (good)      500  100.0%
   (0.70, 1]   (bad)         0    0.0%
    (1, Inf)   (very bad)    0    0.0%

The shape is `(4, 1000, 500)`: chains, draws, observations.
Select the response likelihood, not the total log posterior or its priors.
The distribution must retain per-observation log probabilities.
See {func}`~liesel.model.log_prob_pointwise` for the exact rules.

## Read the result

`elpd_loo` estimates expected log predictive density under leave-one-out
prediction; larger is better on the default log scale. `se` describes
uncertainty in that estimate. Inspect the reported Pareto-k diagnostics and
warnings: unreliable importance weights can make the approximation unsuitable.
Here all 500 Pareto-k values are in the reported good range. `p_loo` is the
estimated effective number of parameters (about 3.07 here); it gives context
for model complexity and is not the predictive score to maximize.
The [ArviZ PSIS-LOO documentation](https://python.arviz.org/en/stable/api/generated/arviz.loo.html) explains
the result fields and diagnostics.

The optional `samples` argument to {func}`~liesel.goose.loo` is retained for
compatibility but ignored. By default, relative MCMC efficiency is estimated
from likelihood values. Use {func}`gs.loo(log_lik_y) <liesel.goose.loo>` for this workflow.

## Compare like with like

Fit each candidate model and evaluate the **same observations**, in the same
order and on the same likelihood scale. Compare their predictive scores with
uncertainty, not just a ranking. The standard error for an individual score
is not the standard error of a paired difference between two models.

The observation axis defines what is left out. Leaving out one response is
a different prediction task from leaving out a group or a future time interval.
Do not flatten dependent likelihood factors into purported independent
observations without checking that the resulting task matches your analysis.